<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Deep_neural_network_of_L_layers/blob/main/L_layered_Deep_learning_from_scrach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [570]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles

X, Y = make_circles(n_samples=6000,noise=0.15,factor=0.5)

In [571]:

print(Y.shape[0] , X.T.shape)


6000 (2, 6000)


In [572]:
def init_parameters(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * 0.01
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b


In [573]:
def init_parameters_he(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * np.sqrt(2/layers_dims[l-1])
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b

In [574]:
def init_parameters_random(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * 0.01
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b


In [575]:
def init_parameters_zeros(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.zeros((layers_dims[l] , layers_dims[l-1]))
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b

In [576]:
def forward(W , b , X , L):
  Zs = []
  As = []
  A = X.T
  As.append(A)

  def relu(Z):
    return np.maximum(0 , Z)
  def sigmoid(Z):
    return 1/(1 + np.exp(-Z))

  for l in range(1 , L-1):
    A_prev = A
    Z = np.dot(W[l],A_prev)+b[l]
    Zs.append(Z)
    A = relu(Z)
    As.append(A)

  ZL = np.dot(W[L-1],A)+b[L-1]
  AL = sigmoid(ZL)
  Zs.append(ZL)
  As.append(AL)
  return Zs , As

In [577]:
def cost(Y , Y_hat):
  m = Y.shape[1]
  temp = Y * np.log(Y_hat) + (1-Y) * np.log(1-Y_hat)
  return -np.sum(temp)/m

In [578]:
def backward(W, Zs, As, Y, L):

    dW = {}

    db = {}

    m = Y.shape[1]

    AL = As[-1]

    dZ = AL - Y

    for l in reversed(range(1, L)):
        dW[l] = (1/m) * np.dot(dZ, As[l-1].T)
        db[l] = (1/m) * np.sum(dZ,axis=1,keepdims=True)
        if l > 1:
            dA = np.dot(W[l].T, dZ)
            dZ = dA * (Zs[l-2] > 0)

    return dW, db

In [579]:
def backward_with_L2(W, Zs, As, Y, L , lambd):

    dW = {}

    db = {}

    m = Y.shape[1]

    AL = As[-1]

    dZ = AL - Y

    for l in reversed(range(1, L)):
        dW[l] = (1/m) * np.dot(dZ, As[l-1].T) + (lambd/m)*W[l]
        db[l] = (1/m) * np.sum(dZ,axis=1,keepdims=True)
        if l > 1:
            dA = np.dot(W[l].T, dZ)
            dZ = dA * (Zs[l-2] > 0)

    return dW, db

In [580]:
def gradient(W , b , X , Y , L , alpha):
  Zs , As = forward(W , b , X , L)
  dW , db = backward(W , Zs , As , Y , L)
  for l in range(1, L):
    W[l] = W[l] - alpha * dW[l]
    b[l] = b[l] - alpha * db[l]
  return W , b

In [581]:
def gradient_with_L2(W , b , X , Y , L , alpha , lambd):
  Zs , As = forward(W , b , X , L)
  dW , db = backward_with_L2(W , Zs , As , Y , L , lambd)
  for l in range(1, L):
    W[l] = W[l] - alpha * dW[l]
    b[l] = b[l] - alpha * db[l]
  return W , b

In [582]:
def L2_cost(W, b, X, Y, L, lambd):

    m = X.shape[1]

    L2 = 0

    for l in range(1, L):
        L2 += np.sum(W[l]**2)

    _, As = forward(W, b, X, L)
    cross = cost(Y, As[-1])
    reg = (lambd/(2*m))*L2

    # print("Cross Entropy:", cross)
    # print("L2 Penalty:", reg)

    # return cross + reg

    return cost(Y, As[-1]) + (lambd/(2*m))*L2

In [583]:
def model(dim , X , Y , initialization , reg , alpha):

  if initialization == "zeros":
        W , b = init_parameters_zeros(dim)
  elif initialization == "random":
        W , b = init_parameters_random(dim)
  elif initialization == "he":
        W , b = init_parameters_he(dim)

  # W , b = init_parameters(dim)
  L = len(dim)

  for i in range(10000):
    if(reg == "L2"):
      W , b = gradient_with_L2(W , b , X , Y , L , alpha , 0.7)
    elif(reg == "normal"):
      W , b = gradient(W , b , X , Y , L , alpha)
    if i % 1000 == 0:
        _, As = forward(W, b, X, L)
        if(reg == "L2"):
          c = L2_cost(W, b, X, Y, L, 0.7)
        elif(reg == "normal"):
          c = cost(Y, As[-1])
        print(f"Iteration {i}: Cost = {c:.4f}")
  return W , b

In [590]:
Y = Y.reshape(1, -1)
W , b = model([2 , 3  , 1] , X , Y , initialization="he" , reg= "L2" , alpha = 0.01)


Iteration 0: Cost = 1.0348
Iteration 1000: Cost = 1.0535
Iteration 2000: Cost = 1.1751
Iteration 3000: Cost = 1.3485
Iteration 4000: Cost = 1.5759
Iteration 5000: Cost = 1.8778
Iteration 6000: Cost = 2.2947
Iteration 7000: Cost = 2.8791
Iteration 8000: Cost = 3.6511
Iteration 9000: Cost = 4.5684


In [592]:
L = len([2 , 3 , 1])
Zs, As = forward(W, b, X, L)
predictions = (As[-1] >= 0.5)
accuracy = np.mean(predictions == Y) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 88.03%


optimization techniques
